# Example: Building a Classical Minimum-Variance Portfolio

In this example, we generate synthetic asset return data, estimate the expected return vector and covariance matrix, solve the minimum-variance optimization problem, compute the efficient frontier, and explore how sensitive the solution is to estimation error. Let's dive in!

> __Learning Objectives:__
>
> * __Synthetic data and estimation:__ Generate a controlled multi-asset return dataset with known ground-truth parameters, then estimate the expected return vector and covariance matrix from the sample. Quantify the estimation error for both the mean and covariance.
> * __Optimization and the efficient frontier:__ Solve the Markowitz minimum-variance portfolio problem and sweep the target return to trace the full efficient frontier. Compare the minimum-variance solution with an equal-weight benchmark.
> * __Sensitivity analysis:__ Demonstrate the error maximization property by perturbing expected returns and visualizing weight instability. Connect the condition number of the covariance matrix to optimizer sensitivity.

## Setup, Data and Prerequisites
We begin by loading our packages and helper functions via the `Include.jl` file. This activates the local Julia environment and loads all dependencies.

In [ ]:
# --- Load packages and helper functions ---
# The Include.jl file activates the local Julia environment and imports all dependencies.
include("Include.jl");

___
## Task 1: Define Our Asset Universe and Generate Synthetic Data
We'll work with a universe of $N = 5$ synthetic assets, loosely inspired by a diversified portfolio: a large-cap equity, a small-cap equity, an international equity, a bond, and a commodity. We define "true" parameters for each asset and use a multivariate normal distribution to generate synthetic daily returns.

> __What are we going to do?__
>
> We'll build the ground-truth parameters step by step: first the expected returns and volatilities, then the correlation structure, then the full covariance matrix $\boldsymbol{\Sigma} = \mathbf{D}\mathbf{C}\mathbf{D}$ where $\mathbf{D} = \text{diag}(\sigma_1,\ldots,\sigma_N)$. Having the true parameters lets us measure _exactly_ how estimation error affects the optimizer.

In later sessions, we'll use [JumpHMM.jl](https://github.com/varnerlab/JumpHMM.jl) to generate more realistic regime-switching synthetic paths. For now, the multivariate normal gives us a clean baseline.

The code below initializes the asset names, defines annualized expected returns and volatilities, converts them to daily units, and displays the results in the `params_df::DataFrame` variable.

In [ ]:
# --- Step 1: Define asset names and true parameters ---
asset_names = ["LargeCap", "SmallCap", "International", "Bond", "Commodity"];
N = length(asset_names);

# "True" annualized expected returns and volatilities
μ_annual = [0.10, 0.12, 0.08, 0.03, 0.06];  # annual returns
σ_annual = [0.18, 0.25, 0.22, 0.05, 0.20];   # annual volatilities

# --- Step 2: Convert annualized parameters to daily ---
# Daily mean: divide by 252 trading days
# Daily volatility: divide by √252
μ_true = μ_annual ./ 252;
σ_daily = σ_annual ./ sqrt(252);

# --- Step 3: Display parameter summary ---
params_df = DataFrame(
    "Asset" => asset_names,
    "μ (annual %)" => μ_annual .* 100,
    "σ (annual %)" => σ_annual .* 100,
    "μ (daily bps)" => round.(μ_true .* 10000, digits=2),
    "σ (daily bps)" => round.(σ_daily .* 10000, digits=2)
);
pretty_table(params_df, tf = tf_markdown)

Next, we define the "true" correlation matrix $\mathbf{C}$. Notice the structure: equities are positively correlated with each other (0.55 to 0.70), the bond is _negatively_ correlated with equities (−0.05 to −0.10), and the commodity has low positive correlations with everything. This is a stylized version of the diversification structure investors rely on.

The code below constructs the `C_true::Matrix{Float64}` correlation matrix and visualizes it as a heatmap.

In [ ]:
# --- Step 1: Define the "true" correlation matrix ---
# Structure: equities are positively correlated, bond is negatively correlated with equities,
# and the commodity has low positive correlations with everything.
C_true = [
    1.00  0.70  0.65  -0.10  0.15;
    0.70  1.00  0.55  -0.05  0.20;
    0.65  0.55  1.00  -0.08  0.25;
   -0.10 -0.05 -0.08   1.00 -0.05;
    0.15  0.20  0.25  -0.05  1.00
];

# --- Step 2: Visualize the correlation matrix as a heatmap ---
heatmap(C_true, xticks=(1:N, asset_names), yticks=(1:N, asset_names),
    title="True Correlation Matrix", color=:RdBu, clims=(-1, 1),
    xrotation=45, size=(450, 400), aspect_ratio=:equal)

Now we construct the full covariance matrix from the correlation matrix and the volatilities: $\boldsymbol{\Sigma} = \mathbf{D}\,\mathbf{C}\,\mathbf{D}$ where $\mathbf{D} = \text{diag}(\sigma_1,\ldots,\sigma_N)$. We verify that the result is symmetric and positive definite, both of which are required for the quadratic program to be well-posed.

The code below computes the `Σ_true::Matrix{Float64}` covariance matrix and runs assertion checks on its properties.

In [ ]:
# --- Step 1: Construct the covariance matrix Σ = D * C * D ---
# D is a diagonal matrix of daily volatilities
D = diagm(σ_daily);
Σ_true = D * C_true * D;

# --- Step 2: Verify required properties ---
# Symmetry and positive definiteness are necessary for the QP to be well-posed
@assert issymmetric(Σ_true) "Covariance matrix must be symmetric"
@assert isposdef(Σ_true) "Covariance matrix must be positive definite"
println("✓ Σ_true is symmetric: $(issymmetric(Σ_true))")
println("✓ Σ_true is positive definite: $(isposdef(Σ_true))")
println("✓ Condition number: $(round(cond(Σ_true), digits=1))")

With the ground-truth parameters in hand, we generate $T = 504$ trading days (approximately 2 years) of synthetic daily returns from a [multivariate normal distribution](https://juliastats.org/Distributions.jl/stable/multivariate/#Distributions.MvNormal) $\mathbf{r}_t \sim \mathcal{N}(\boldsymbol{\mu}, \boldsymbol{\Sigma})$ and save the data for use in the stress-testing example.

The code below generates the synthetic returns, stores them in the `df_returns::DataFrame` variable, and saves the data to disk using [the `save(...)` function](https://juliaio.github.io/JLD2.jl/stable/).

In [ ]:
# --- Step 1: Generate synthetic daily returns ---
# T = 504 trading days ≈ 2 years of daily data
T = 504;
dist = MvNormal(μ_true, Σ_true);       # multivariate normal with true parameters
returns_matrix = rand(dist, T)';        # T × N matrix (transpose to get rows = days)

# --- Step 2: Store returns in a DataFrame ---
df_returns = DataFrame(returns_matrix, asset_names);

# --- Step 3: Save data to disk for reuse in the stress-testing example ---
save(joinpath(_PATH_TO_DATA, "synthetic-returns.jld2"), 
    Dict("returns" => df_returns, "asset_names" => asset_names,
         "mu_true" => μ_true, "Sigma_true" => Σ_true, "C_true" => C_true,
         "sigma_daily" => σ_daily));

# --- Step 4: Display summary statistics ---
println("Generated $(T) days × $(N) assets of synthetic return data.\n")
summary_df = DataFrame(
    "Asset" => asset_names,
    "Mean (daily bps)" => round.(vec(mean(returns_matrix, dims=1)) .* 10000, digits=2),
    "Std (daily bps)" => round.(vec(std(returns_matrix, dims=1)) .* 10000, digits=2),
    "Min (%)" => round.(vec(minimum(returns_matrix, dims=1)) .* 100, digits=3),
    "Max (%)" => round.(vec(maximum(returns_matrix, dims=1)) .* 100, digits=3),
);
pretty_table(summary_df, tf = tf_markdown)

___
## Task 2: Estimate Return and Covariance from the Synthetic Data
In practice, we don't know the true $\boldsymbol{\mu}$ and $\boldsymbol{\Sigma}$. We estimate them from historical data. Let's compute the sample estimates and measure the estimation error.

> __What are we going to do?__
>
> We'll compute the sample mean $\hat{\boldsymbol{\mu}} = \frac{1}{T}\sum_{t=1}^{T}\mathbf{r}_t$ and sample covariance $\hat{\boldsymbol{\Sigma}} = \frac{1}{T-1}\sum_{t=1}^{T}(\mathbf{r}_t - \hat{\boldsymbol{\mu}})(\mathbf{r}_t - \hat{\boldsymbol{\mu}})^{\top}$, compare them to the true values, visualize the correlation estimation error, and examine the eigenstructure of the covariance matrix.

With 504 observations (2 years of daily data), expected return estimates will have noticeable error bars (returns are hard to estimate), while covariance estimates will be more stable. This asymmetry is key to understanding why the optimizer misbehaves.

The code below computes the sample mean `μ_hat::Vector{Float64}` and sample covariance `Σ_hat::Matrix{Float64}`, then compares the estimated returns to the true values.

In [ ]:
# --- Step 1: Extract the return matrix ---
R = Matrix(df_returns); # T × N matrix of daily returns

# --- Step 2: Compute sample mean and covariance ---
μ_hat = vec(mean(R, dims=1));  # sample mean vector (N × 1)
Σ_hat = cov(R);                # sample covariance matrix (N × N)

# --- Step 3: Verify symmetry of the sample covariance ---
@assert isapprox(Σ_hat, Σ_hat', rtol=1e-10) "Sample covariance must be symmetric"

# --- Step 4: Compare true vs estimated expected returns (annualized) ---
comparison = DataFrame(
    "Asset" => asset_names,
    "μ_true (annual %)" => round.(μ_true .* 252 .* 100, digits=2),
    "μ_hat (annual %)" => round.(μ_hat .* 252 .* 100, digits=2),
    "Error (bps)" => round.((μ_hat .- μ_true) .* 252 .* 10000, digits=1)
);

println("True vs. Estimated Expected Returns:")
pretty_table(comparison, tf = tf_markdown)

Let's compare the true and estimated correlation matrices side by side, and also look at the _estimation error_ matrix $\hat{\mathbf{C}} - \mathbf{C}$.

> __What should you see?__
>
> The estimated correlations should be close to the true values, but with some noise. The error matrix reveals which pairwise correlations are most affected by estimation error. These are the ones the optimizer will exploit most aggressively.

The `let...end` block below computes the estimated correlation matrix, the error matrix, and plots all three as heatmaps.

In [ ]:
let
    # --- Step 1: Compute estimated correlation and error matrices ---
    C_hat = cor(R);              # sample correlation matrix
    C_error = C_hat .- C_true;   # element-wise estimation error

    # --- Step 2: Plot three panels: true, estimated, and error ---
    p1 = heatmap(C_true, title="True C", xticks=(1:N, asset_names), 
        yticks=(1:N, asset_names), clims=(-1,1), color=:RdBu, xrotation=45);
    p2 = heatmap(C_hat, title="Estimated Ĉ", xticks=(1:N, asset_names), 
        yticks=(1:N, asset_names), clims=(-1,1), color=:RdBu, xrotation=45);
    p3 = heatmap(C_error, title="Error (Ĉ − C)", xticks=(1:N, asset_names), 
        yticks=(1:N, asset_names), clims=(-0.15,0.15), color=:RdBu, xrotation=45);
    
    plot(p1, p2, p3, layout=(1,3), size=(1100,350))
end

**Eigenvalue Analysis:** The eigenvalues of $\hat{\boldsymbol{\Sigma}}$ reveal the _effective dimensionality_ of the risk. If a few eigenvalues dominate, most of the portfolio variance can be explained by a small number of principal components. The condition number $\kappa = \lambda_{\max}/\lambda_{\min}$ measures how sensitive the optimization will be to perturbations.

> __What should you see?__
>
> A high condition number means the optimizer is working in a poorly conditioned space, where small changes in inputs produce large changes in the solution. This is one of the mathematical reasons behind the "error maximization" property.

The `let...end` block below computes the eigenvalue spectra of both the true and estimated covariance matrices, reports the condition numbers, and plots the eigenvalue comparison.

In [ ]:
let
    # --- Step 1: Compute eigenvalues (sorted largest to smallest) ---
    λ_true = eigvals(Σ_true) |> sort |> reverse;
    λ_hat = eigvals(Σ_hat) |> sort |> reverse;
    
    # --- Step 2: Compute condition numbers κ = λ_max / λ_min ---
    κ_true = λ_true[1] / λ_true[end];
    κ_hat = λ_hat[1] / λ_hat[end];
    
    # --- Step 3: Plot eigenvalue spectrum comparison ---
    p = plot(1:N, λ_true .* 10000, marker=:circle, label="True Σ", lw=2, 
        xlabel="Principal Component", ylabel="Eigenvalue (×10⁴)", 
        title="Eigenvalue Spectrum of Covariance Matrix");
    plot!(p, 1:N, λ_hat .* 10000, marker=:diamond, label="Estimated Σ̂", lw=2, 
        linestyle=:dash, size=(600, 400));
    
    # --- Step 4: Report condition numbers and variance explained ---
    println("Condition number (true):      $(round(κ_true, digits=1))")
    println("Condition number (estimated): $(round(κ_hat, digits=1))")
    println("\nVariance explained by PC1: $(round(λ_hat[1]/sum(λ_hat)*100, digits=1))%")
    println("Variance explained by PC1+PC2: $(round(sum(λ_hat[1:2])/sum(λ_hat)*100, digits=1))%")
    
    p
end

___
## Task 3: Solve the Minimum-Variance Problem, Compute the Efficient Frontier, and Test Input Sensitivity
Now we solve the classical minimum-variance problem using the _estimated_ parameters (not the true ones). We'll then sweep the target return $R$ to trace out the entire efficient frontier, and finally test the "error maximization" property by perturbing expected returns.

> __What are we going to do?__
>
> Three sub-steps: (1) solve for a single target return and inspect the weights, (2) sweep $R$ from 0 to the maximum feasible return to compute the efficient frontier, and (3) perturb the LargeCap expected return by $\pm 100$ bps and visualize how weights shift.

The optimizer will concentrate weight in the Bond (lowest variance). The efficient frontier will show the risk-return trade-off. The sensitivity test will reveal weight instability from small input changes.

The code below uses [the `build(...)` function](../../code/docs/build/session1.html) to construct a [`MyPortfolioAllocationProblem`](../../code/docs/build/session1.html) instance, then calls [the `solve_minvariance(...)` function](../../code/docs/build/session1.html) to find the optimal weights. The results are stored in the `result::MyPortfolioPerformanceResult` variable.

In [ ]:
# --- Step 1: Define portfolio constraints ---
# Long-only constraints: 0 ≤ wᵢ ≤ 1 for each asset
bounds = hcat(zeros(N), ones(N));
R_target = 0.05 / 252;  # target: 5% annualized, converted to daily

# --- Step 2: Build the allocation problem and solve ---
# build(...) constructs a MyPortfolioAllocationProblem with the estimated parameters
problem = build(MyPortfolioAllocationProblem;
    μ = μ_hat, Σ = Σ_hat, bounds = bounds, R = R_target);

# solve_minvariance(...) solves the quadratic program for minimum variance
result = solve_minvariance(problem);

# --- Step 3: Verify constraints are satisfied ---
@assert isapprox(sum(result.weights), 1.0, rtol=1e-4) "Weights must sum to 1"
@assert all(result.weights .>= -1e-8) "Weights must be non-negative (long-only)"

# --- Step 4: Display portfolio weights and performance ---
weights_df = DataFrame(
    "Asset" => asset_names,
    "Weight (%)" => round.(result.weights .* 100, digits=2)
);

println("Minimum-Variance Portfolio Weights (target return = 5% annual):")
println("═"^55)
pretty_table(weights_df, tf = tf_markdown)
println("\nPortfolio expected return (annual): $(round(result.expected_return * 252 * 100, digits=2))%")
println("Portfolio volatility (annual):     $(round(sqrt(result.variance) * sqrt(252) * 100, digits=2))%")

# --- Step 5: Save baseline portfolio for later examples ---
save(joinpath(_PATH_TO_DATA, "baseline-portfolio.jld2"),
    Dict("weights" => result.weights, "mu_hat" => μ_hat, "Sigma_hat" => Σ_hat,
         "asset_names" => asset_names, "result" => result));

Let's visualize the portfolio weight allocation as a bar chart.

> __What should you see?__
>
> The Bond asset will likely dominate the allocation because of its much lower variance. This is the _concentration problem_ in action. The optimizer loads low-variance assets regardless of whether that concentration is desirable from a diversification standpoint.

The `let...end` block below loads the saved baseline portfolio and plots the weights as a bar chart.

In [ ]:
let
    # --- Step 1: Load the baseline portfolio from disk ---
    data = load(joinpath(_PATH_TO_DATA, "baseline-portfolio.jld2"));
    weights = data["weights"];
    asset_names = data["asset_names"];

    # --- Step 2: Bar chart of portfolio weights ---
    bar(asset_names, weights .* 100, 
        ylabel="Weight (%)", xlabel="Asset", title="Minimum-Variance Portfolio Weights",
        legend=false, color=:steelblue, bar_width=0.6, size=(600,400),
        ylims=(0, 100))
end

**Efficient Frontier:** Now we sweep the target return $R$ from 0 to the maximum feasible value and solve the quadratic program at each point. This traces out the _efficient frontier_, the set of lowest-risk portfolios for every attainable return level.

> __What should you see?__
>
> The frontier curves upward and to the right: higher expected return requires accepting more risk. Our baseline portfolio (the red star) sits at one point on this curve. The equal-weight portfolio (green diamond) will typically lie _below_ the frontier, indicating it is suboptimal in the Markowitz sense.

The `let...end` block below sweeps the target return, solves the QP at each point using [the `solve_minvariance(...)` function](../../code/docs/build/session1.html), and plots the efficient frontier with the baseline and equal-weight portfolios marked.

In [ ]:
let
    # --- Step 1: Define the sweep range for target returns ---
    R_min = 0.0;
    R_max = maximum(μ_hat) * 0.95;  # just below the max feasible return
    n_points = 100;
    R_sweep = range(R_min, stop=R_max, length=n_points) |> collect;
    
    # --- Step 2: Solve the QP at each target return ---
    frontier_risk = Float64[];
    frontier_return = Float64[];
    
    for R_i ∈ R_sweep
        try
            prob = build(MyPortfolioAllocationProblem;
                μ = μ_hat, Σ = Σ_hat, bounds = bounds, R = R_i);
            sol = solve_minvariance(prob);
            
            σ_p = sqrt(sol.variance) * sqrt(252) * 100;  # annualized volatility %
            μ_p = sol.expected_return * 252 * 100;         # annualized return %
            
            push!(frontier_risk, σ_p);
            push!(frontier_return, μ_p);
        catch
            # infeasible target return, skip this point
        end
    end
    
    # --- Step 3: Compute the baseline portfolio's position on the frontier ---
    baseline_σ = sqrt(result.variance) * sqrt(252) * 100;
    baseline_μ = result.expected_return * 252 * 100;
    
    # --- Step 4: Compute equal-weight portfolio for comparison ---
    w_eq = fill(1.0/N, N);                          # equal weights: 1/N each
    eq_var = dot(w_eq, Σ_hat * w_eq);               # portfolio variance
    eq_ret = dot(μ_hat, w_eq);                       # portfolio expected return
    eq_σ = sqrt(eq_var) * sqrt(252) * 100;           # annualized volatility %
    eq_μ = eq_ret * 252 * 100;                       # annualized return %
    
    # --- Step 5: Plot the efficient frontier ---
    p = plot(frontier_risk, frontier_return, lw=2, color=:steelblue, label="Efficient Frontier",
        xlabel="Volatility (annual %)", ylabel="Expected Return (annual %)",
        title="Efficient Frontier", legend=:topleft, size=(700, 450));
    scatter!(p, [baseline_σ], [baseline_μ], marker=:star5, ms=12, color=:red, 
        label="Our Portfolio (R=5%)");
    scatter!(p, [eq_σ], [eq_μ], marker=:diamond, ms=8, color=:green, 
        label="Equal-Weight");
    
    p
end

**Input Sensitivity Test:** Let's test the "error maximization" property directly. We perturb the expected return estimate for LargeCap by $\pm 100$ basis points and re-solve each time.

> __What should you see?__
>
> Even modest perturbations to a single asset's expected return can shift weights significantly. This demonstrates that the optimizer is highly sensitive to the _least reliable_ input, the estimated expected returns.

The `let...end` block below perturbs the LargeCap expected return at five levels, solves the QP at each, and displays the resulting weight allocations in a table.

In [ ]:
let
    # --- Step 1: Load saved baseline data ---
    data = load(joinpath(_PATH_TO_DATA, "baseline-portfolio.jld2"));
    μ_hat = data["mu_hat"];
    Σ_hat = data["Sigma_hat"];
    asset_names = data["asset_names"];
    baseline_weights = data["weights"];
    N = length(asset_names);
    
    # --- Step 2: Define constraints and target return ---
    bounds = hcat(zeros(N), ones(N));
    R_target = 0.05 / 252;
    
    # --- Step 3: Perturb LargeCap (index 1) expected return by ±100 bps annually ---
    perturbations_bps = [-100, -50, 0, 50, 100];
    results = DataFrame();

    for δ_bps in perturbations_bps
        
        # Apply perturbation: convert annual bps to daily
        μ_perturbed = copy(μ_hat);
        μ_perturbed[1] += (δ_bps / 10000) / 252;

        # Solve the perturbed QP
        problem = build(MyPortfolioAllocationProblem;
            μ = μ_perturbed, Σ = Σ_hat, bounds = bounds, R = R_target);
        result = solve_minvariance(problem);

        # Store the row of weights
        row = DataFrame("Perturbation (bps)" => δ_bps, 
            [name => round(w * 100, digits=2) for (name, w) in zip(asset_names, result.weights)]...);
        results = vcat(results, row);
    end

    # --- Step 4: Display the sensitivity table ---
    println("LargeCap expected return perturbation → weight sensitivity:")
    pretty_table(results, tf=tf_markdown)
end

Let's plot the weight sensitivity as a grouped bar chart. Each cluster of bars shows how one asset's weight changes across the perturbation levels.

The `let...end` block below recomputes the perturbed weights and stores them in the `weight_matrix::Matrix{Float64}` variable for plotting.

In [ ]:
let
    # --- Step 1: Define perturbation levels ---
    perturbations_bps = [-100, -50, 0, 50, 100];
    weight_matrix = zeros(length(perturbations_bps), N);
    
    # --- Step 2: Solve the QP at each perturbation level ---
    for (j, δ_bps) ∈ enumerate(perturbations_bps)
        μ_perturbed = copy(μ_hat);
        μ_perturbed[1] += (δ_bps / 10000) / 252;  # convert annual bps to daily
        
        prob = build(MyPortfolioAllocationProblem;
            μ = μ_perturbed, Σ = Σ_hat, bounds = bounds, R = R_target);
        sol = solve_minvariance(prob);
        weight_matrix[j, :] = sol.weights .* 100;  # store as percentages
    end
    
    # --- Step 3: Grouped bar chart of weight sensitivity ---
    groupedbar(asset_names, weight_matrix', 
        bar_position=:dodge, bar_width=0.15,
        label=permutedims(["$(δ) bps" for δ ∈ perturbations_bps]),
        ylabel="Weight (%)", xlabel="Asset",
        title="Weight Sensitivity to LargeCap μ Perturbation",
        size=(750, 450), legend=:topright)
end

___
## Summary
The minimum-variance optimizer concentrates weight in the lowest-variance asset and is highly sensitive to estimation error in expected returns, confirming the classical "error maximization" diagnosis.
These limitations motivate the regularization and robust optimization techniques explored in later sessions.

### Key Takeaways

* **Concentration risk:** The minimum-variance optimizer heavily loads the Bond asset (lowest variance), producing a portfolio that is optimal on paper but poorly diversified. The efficient frontier shows where this portfolio sits relative to alternatives such as the equal-weight benchmark.
* **Asymmetric estimation error:** The covariance matrix is estimated well from 2 years of daily data, but the expected return vector remains noisy. The optimizer exploits this noise aggressively, as reflected by the covariance condition number.
* **Input sensitivity:** Perturbing LargeCap's expected return by just 100 basis points produces large weight shifts. This confirms that small estimation errors in expected returns propagate into large changes in the optimal allocation.

In the [next example](eCornell-AI-Finance-S1-Example-StressTestMinVariancePortfolio-May-2026.ipynb), we'll stress-test this baseline portfolio under correlation shifts, price drops, and higher trading costs.

### Disclaimer
This content is for educational purposes only and does not constitute investment advice. The examples use synthetic data and simplified models.